# IBDD analysis

What the Image-Based Drift Detector saw during an `ovr.py` run with `"ibdd"` in `DRIFT_DETECTORS`. The run must have `SAVE_DETECTOR_TRACE` and `SAVE_IBDD_IMAGES` on.

IBDD draws a window of `w` examples as a grayscale image, with one row per feature and one column per example. It compares that image with a **reference image** of the training data through the Mean-Squared Deviation (MSD). The thresholds are `mean ± 2·std` of the MSDs between the reference and 20 random training windows (the **calibration**). A test batch is flagged as drift when its MSD falls outside `[lower, upper]`.

Files read from `RESULTS_DIR/<dataset>/`:
- `distances_ibdd.csv`: the calibration MSDs and the thresholds.
- `detector_trace.csv`: per batch, the MSD, the thresholds and the drift flag (rows with `detector == "ibdd"`).
- `ibdd/reference.npz`: the training reference image.
- `ibdd/batch_XXXX.npz`: each test batch's image exactly as IBDD compared it (after its fixed-seed shuffle), its true labels in image-column order, the MSD and the flag.

Sections:
1. The calibration MSD distribution with its thresholds, plus a rug of the test-batch MSDs.
2. The per-batch MSD timeline, with drift batches in red.
3. An image viewer showing the reference, the batch and `|batch − reference|`. Flagged batches get a red frame and a **DRIFT** title.
4. A contact sheet of batch thumbnails, with the flagged ones framed in red.

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

# Repository root (the folder holding ovr.py), found from wherever the notebook runs.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ovr.py").exists())
# ovr.py run folder (its RESULTS_ROOT) to inspect.
RESULTS_DIR = ROOT / "results" / "ovr_results_cdt_ibdd_ours"

DRIFT_COLOR = "crimson"
STABLE_COLOR = "slategray"
CALIBRATION_COLOR = "royalblue"

# Every dataset of the run for which IBDD was fitted.
DATASETS = sorted(p.parent.name for p in RESULTS_DIR.glob("*/distances_ibdd.csv"))
if not DATASETS:
    raise FileNotFoundError(f"No */distances_ibdd.csv found under {RESULTS_DIR}")


def dataset_dir(dataset):
    return RESULTS_DIR / dataset


def load_calibration(dataset):
    """Calibration MSDs and the thresholds fitted from them (distances_ibdd.csv)."""
    row = pd.read_csv(dataset_dir(dataset) / "distances_ibdd.csv").iloc[-1]
    return np.asarray(json.loads(row["distances"]), dtype=float), float(row["thr_lower"]), float(row["thr_upper"])


def load_trace(dataset):
    """Per-batch IBDD rows of detector_trace.csv (empty frame when missing)."""
    path = dataset_dir(dataset) / "detector_trace.csv"
    if not path.exists():
        return pd.DataFrame(columns=["batch_index", "statistic", "thr_lower", "thr_upper", "drift"])
    trace = pd.read_csv(path)
    return trace[trace["detector"] == "ibdd"].sort_values("batch_index").reset_index(drop=True)


def load_reference(dataset):
    with np.load(dataset_dir(dataset) / "ibdd" / "reference.npz") as data:
        return {key: data[key] for key in data.files}


def list_batches(dataset):
    return sorted(int(p.stem.split("_")[1]) for p in (dataset_dir(dataset) / "ibdd").glob("batch_*.npz"))


def load_batch(dataset, batch_index):
    # allow_pickle: older runs stored string labels as an object array.
    with np.load(dataset_dir(dataset) / "ibdd" / f"batch_{batch_index:04d}.npz", allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


def gray(image):
    """Channel 0 of a JPEG-backend image (the Greys colormap gives R = G = B up
    to JPEG noise) or the array-backend image itself, as float."""
    image = np.asarray(image, dtype=float)
    return image[..., 0] if image.ndim == 3 else image


for dataset in DATASETS:
    distances, lower, upper = load_calibration(dataset)
    trace = load_trace(dataset)
    reference = load_reference(dataset)["reference_image"] if (dataset_dir(dataset) / "ibdd" / "reference.npz").exists() else None
    print(
        f"{dataset}: thresholds [{lower:.1f}, {upper:.1f}] from {distances.size} calibration MSDs | "
        f"{len(trace)} traced batches, {int(trace['drift'].sum())} flagged | "
        f"{len(list_batches(dataset)) if reference is not None else 0} saved images"
        + (f" | image {reference.shape[0]} features x {reference.shape[1]} examples" if reference is not None else "")
    )

## 1. Calibration MSDs and thresholds

The KDE shows the calibration MSDs between the reference image and the random training windows. The dashed and dotted lines are the upper and lower thresholds (`mean ± 2·std`, lower clipped at 0). Below the axis, a rug marks every test batch's MSD: gray is inside the thresholds, red is flagged as drift.

In [ ]:
def make_calibration_figure(dataset):
    distances, lower, upper = load_calibration(dataset)
    trace = load_trace(dataset)
    test_msd = trace["statistic"].to_numpy(dtype=float)

    values = np.concatenate([distances, test_msd, [lower, upper]])
    pad = 0.05 * (values.max() - values.min() or 1.0)
    grid = np.linspace(values.min() - pad, values.max() + pad, 400)
    jitter = distances if np.std(distances) > 0 else distances + np.random.default_rng(0).normal(0, 1e-6, distances.size)
    density = gaussian_kde(jitter)(grid)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=grid, y=density, mode="lines", name="calibration MSD (KDE)",
                             line=dict(color=CALIBRATION_COLOR, width=2.5), fill="tozeroy"))
    fig.add_trace(go.Scatter(x=distances, y=np.zeros_like(distances), mode="markers", name="calibration windows",
                             marker=dict(symbol="line-ns-open", size=14, color=CALIBRATION_COLOR)))
    rug_y = -0.08 * density.max()
    for flagged, color, name in [(False, STABLE_COLOR, "test batch (no drift)"), (True, DRIFT_COLOR, "test batch (DRIFT)")]:
        part = trace[trace["drift"] == flagged]
        fig.add_trace(go.Scatter(
            x=part["statistic"], y=np.full(len(part), rug_y), mode="markers", name=name,
            marker=dict(symbol="line-ns-open", size=16, color=color),
            customdata=part["batch_index"], hovertemplate="batch %{customdata}<br>MSD=%{x:.2f}<extra></extra>",
        ))
    for value, bound, dash in [(upper, "upper", "dash"), (lower, "lower", "dot")]:
        fig.add_vline(x=value, line=dict(color=DRIFT_COLOR, width=1.5, dash=dash),
                      annotation_text=f"thr {bound} = {value:.1f}", annotation_position="top",
                      annotation_font_color=DRIFT_COLOR)
    fig.update_layout(
        template="plotly_white", height=460,
        title=f"IBDD calibration MSD distribution — {dataset}",
        xaxis_title="MSD(reference, window)", yaxis_title="Density",
        legend=dict(orientation="h", y=-0.2),
    )
    return fig


calibration_dataset = widgets.Dropdown(options=DATASETS, description="Dataset:")
calibration_out = widgets.Output()


def show_calibration(*_):
    with calibration_out:
        calibration_out.clear_output(wait=True)
        make_calibration_figure(calibration_dataset.value).show()


calibration_dataset.observe(show_calibration, names="value")
show_calibration()
display(widgets.VBox([calibration_dataset, calibration_out]))

## 2. Per-batch MSD timeline

The MSD of every test batch, in batch order. For the synthetic datasets the x-axis is the bag time `t`. The shaded band spans `[lower, upper]`, and red markers are the batches IBDD flagged as drift. Those are the batches where every `{base}_ibdd` quantifier switched to its synthetic version.

In [ ]:
def make_timeline_figure(dataset):
    trace = load_trace(dataset)
    _, lower, upper = load_calibration(dataset)
    x_col = "t" if "t" in trace.columns and trace["t"].notna().all() else "batch_index"
    x = trace[x_col]

    fig = go.Figure()
    fig.add_hrect(y0=lower, y1=upper, fillcolor=CALIBRATION_COLOR, opacity=0.10, line_width=0,
                  annotation_text="no-drift band", annotation_position="top left")
    for value, dash in [(upper, "dash"), (lower, "dot")]:
        fig.add_hline(y=value, line=dict(color=DRIFT_COLOR, width=1.2, dash=dash))
    fig.add_trace(go.Scatter(x=x, y=trace["statistic"], mode="lines", name="MSD",
                             line=dict(color=STABLE_COLOR, width=1.2), hoverinfo="skip"))
    for flagged, color, size, name in [(False, STABLE_COLOR, 6, "no drift"), (True, DRIFT_COLOR, 11, "DRIFT")]:
        part = trace[trace["drift"] == flagged]
        fig.add_trace(go.Scatter(
            x=part[x_col], y=part["statistic"], mode="markers", name=name,
            marker=dict(color=color, size=size, symbol="x" if flagged else "circle"),
            customdata=part["batch_index"],
            hovertemplate="batch %{customdata}<br>" + x_col + "=%{x}<br>MSD=%{y:.2f}<extra></extra>",
        ))
    fig.update_layout(
        template="plotly_white", height=420,
        title=f"IBDD MSD per test batch — {dataset} ({int(trace['drift'].sum())}/{len(trace)} flagged)",
        xaxis_title=x_col, yaxis_title="MSD(reference, batch)",
    )
    return fig


timeline_dataset = widgets.Dropdown(options=DATASETS, description="Dataset:")
timeline_out = widgets.Output()


def show_timeline(*_):
    with timeline_out:
        timeline_out.clear_output(wait=True)
        make_timeline_figure(timeline_dataset.value).show()


timeline_dataset.observe(show_timeline, names="value")
show_timeline()
display(widgets.VBox([timeline_dataset, timeline_out]))

## 3. Image viewer: reference vs. batch

The panels, from top to bottom:
- **Training reference**: the last `w` training examples.
- **Test batch**: the batch as IBDD compared it.
- **|batch − reference|**: the pixel-wise deviation the MSD averages. Bright cells show which features (rows) and examples (columns) move the MSD.
- **Batch labels**: the true class of each column.

Images are shown on the grayscale pixel values IBDD compares (0–255 for the JPEG backend). With the `Greys` colormap, darker means a higher feature value.

When a batch is **flagged as drift**, its panel gets a red frame and the title reads `DRIFT — MSD ∉ [lower, upper]`. Use **Jump to flagged** to go straight to those batches. **Order columns by class** applies the same column reordering to all panels, so the pixel pairs being compared stay aligned and class blocks become visible. The MSD itself does not depend on the column order within these aligned pairs.

In [ ]:
LABEL_COLORS = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#EECA3B", "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC"]


def discrete_colorscale(n):
    """Step colorscale mapping the integer codes 0..n-1 to LABEL_COLORS."""
    if n <= 1:
        return [[0.0, LABEL_COLORS[0]], [1.0, LABEL_COLORS[0]]]
    scale = []
    for code in range(n):
        color = LABEL_COLORS[code % len(LABEL_COLORS)]
        scale += [[code / n, color], [(code + 1) / n, color]]
    return scale


def make_image_figure(dataset, batch_index, order_by_class=False):
    reference = load_reference(dataset)
    batch = load_batch(dataset, batch_index)
    lower, upper = float(reference["thr_lower"]), float(reference["thr_upper"])
    ref_img, batch_img = gray(reference["reference_image"]), gray(batch["image"])
    diff = np.abs(batch_img - ref_img)
    labels = batch["labels"]
    msd_value, drift = float(batch["msd"]), bool(batch["drift"])

    order = np.argsort(labels, kind="stable") if order_by_class else np.arange(labels.size)
    ref_img, batch_img, diff, labels = ref_img[:, order], batch_img[:, order], diff[:, order], labels[order]
    classes, codes = np.unique(labels, return_inverse=True)
    n_features = ref_img.shape[0]

    batch_title = (f"<b>DRIFT</b> — batch {batch_index}: MSD={msd_value:.1f} ∉ [{lower:.1f}, {upper:.1f}]" if drift
                   else f"batch {batch_index}: MSD={msd_value:.1f} ∈ [{lower:.1f}, {upper:.1f}]")
    panel_height = int(np.clip(4 * n_features, 70, 320))
    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        row_heights=[panel_height, panel_height, panel_height, 28],
        subplot_titles=["training reference", batch_title, "|batch − reference|", "batch labels (true class)"],
    )
    pixel_range = dict(zmin=0, zmax=255) if reference["reference_image"].dtype == np.uint8 else dict(zmin=0, zmax=1)
    for row, image, name in [(1, ref_img, "reference"), (2, batch_img, "batch")]:
        fig.add_trace(go.Heatmap(z=image, colorscale="gray", showscale=False, name=name, **pixel_range,
                                 hovertemplate="example %{x}<br>feature %{y}<br>pixel %{z:.0f}<extra>" + name + "</extra>"),
                      row=row, col=1)
    fig.add_trace(go.Heatmap(z=diff, colorscale="Reds", zmin=0, zmax=pixel_range["zmax"], name="|diff|",
                             colorbar=dict(title="|diff|", len=0.25, y=0.38),
                             hovertemplate="example %{x}<br>feature %{y}<br>|diff| %{z:.0f}<extra></extra>"),
                  row=3, col=1)
    fig.add_trace(go.Heatmap(z=codes[None, :], colorscale=discrete_colorscale(len(classes)), zmin=0, zmax=max(len(classes), 1),
                             showscale=False, text=[[str(c) for c in labels]],
                             hovertemplate="example %{x}<br>class %{text}<extra></extra>"),
                  row=4, col=1)
    for row in (1, 2, 3):
        fig.update_yaxes(title_text="feature", autorange="reversed", dtick=max(1, n_features // 8), row=row, col=1)
    fig.update_yaxes(showticklabels=False, row=4, col=1)
    fig.update_xaxes(title_text="example (image column)" + (", ordered by class" if order_by_class else ""), row=4, col=1)

    if drift:
        # Red frame around the flagged batch panel and a red panel title.
        fig.add_shape(type="rect", xref="x2 domain", yref="y2 domain", x0=-0.005, x1=1.005, y0=-0.02, y1=1.02,
                      line=dict(color=DRIFT_COLOR, width=5))
        fig.layout.annotations[1].font = dict(color=DRIFT_COLOR, size=15)

    legend = "  ".join(f"<span style='color:{LABEL_COLORS[i % len(LABEL_COLORS)]}'>■</span> {c}" for i, c in enumerate(classes))
    fig.update_layout(
        template="plotly_white", height=3 * panel_height + 330,
        title=dict(text=f"IBDD images — {dataset}, batch {batch_index}" + (" — <b>DRIFT</b>" if drift else "")
                   + f"<br><sup>classes: {legend}</sup>",
                   font=dict(color=DRIFT_COLOR if drift else None)),
        margin=dict(t=110),
    )
    return fig


image_dataset = widgets.Dropdown(options=DATASETS, description="Dataset:")
image_batch = widgets.SelectionSlider(options=[0], description="Batch:", continuous_update=False,
                                      layout=widgets.Layout(width="60%"))
image_flagged = widgets.Dropdown(options=[("—", None)], description="Jump to flagged:",
                                 style={"description_width": "initial"})
image_order = widgets.Checkbox(value=False, description="Order columns by class")
image_out = widgets.Output()


def show_image(*_):
    with image_out:
        image_out.clear_output(wait=True)
        make_image_figure(image_dataset.value, image_batch.value, image_order.value).show()


def on_image_dataset(*_):
    batches = list_batches(image_dataset.value)
    flagged = [int(i) for i in load_trace(image_dataset.value).query("drift")["batch_index"] if int(i) in batches]
    image_batch.unobserve(show_image, names="value")
    image_batch.options = batches
    image_batch.value = batches[0]
    image_batch.observe(show_image, names="value")
    image_flagged.options = [("—", None)] + [(f"batch {i}", i) for i in flagged]
    show_image()


def on_image_flagged(change):
    if change["new"] is not None:
        image_batch.value = change["new"]


image_batch.observe(show_image, names="value")
image_dataset.observe(on_image_dataset, names="value")
image_flagged.observe(on_image_flagged, names="value")
image_order.observe(show_image, names="value")
on_image_dataset()
display(widgets.VBox([widgets.HBox([image_dataset, image_flagged, image_order]), image_batch, image_out]))

## 4. Contact sheet

Thumbnails of the first `max_batches` batch images of each dataset, in batch order. The flagged batches get a thick red frame and a red title.

In [ ]:
def contact_sheet(dataset, max_batches=40, cols=8):
    batches = list_batches(dataset)[:max_batches]
    if not batches:
        print(f"{dataset}: no saved batch images")
        return None
    reference = load_reference(dataset)
    vmax = 255 if reference["reference_image"].dtype == np.uint8 else 1
    rows = int(np.ceil((len(batches) + 1) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 1.6 * rows), squeeze=False)
    tiles = [("reference", gray(reference["reference_image"]), False, None)] + [
        (f"#{i}", gray(b["image"]), bool(b["drift"]), float(b["msd"])) for i, b in ((i, load_batch(dataset, i)) for i in batches)
    ]
    for ax in axes.flat:
        ax.axis("off")
    for ax, (name, image, drift, msd_value) in zip(axes.flat, tiles):
        ax.axis("on")
        ax.imshow(image, cmap="gray", vmin=0, vmax=vmax, aspect="auto", interpolation="nearest")
        ax.set_xticks([]), ax.set_yticks([])
        title = name if msd_value is None else f"{name} MSD={msd_value:.0f}" + (" DRIFT" if drift else "")
        ax.set_title(title, fontsize=8, color=DRIFT_COLOR if drift else "black", fontweight="bold" if drift else "normal")
        for spine in ax.spines.values():
            spine.set_edgecolor(DRIFT_COLOR if drift else ("royalblue" if msd_value is None else "lightgray"))
            spine.set_linewidth(4 if drift else 1.5)
    n_flagged = sum(t[2] for t in tiles)
    fig.suptitle(f"IBDD batch images — {dataset} (first {len(batches)} batches, {n_flagged} flagged)", fontsize=12)
    fig.tight_layout()
    return fig


for dataset in DATASETS:
    if contact_sheet(dataset) is not None:
        plt.show()